In [1]:
import os
import json
import random
from typing import List, Dict, Any

In [2]:
import os
import json
import random
from typing import List, Dict, Any
def format_nl_premises(premises_nl: List[str]) -> str:
    """Format premises as a numbered list string."""
    return "\n".join(f"{i + 1}. {premise}" for i, premise in enumerate(premises_nl))
def format_fol_premises(premises_fol: List[str]) -> str:
    """Format FOL premises as a numbered list string."""
    return "\n".join(f"{i + 1}. {premise}" for i, premise in enumerate(premises_fol))
def build_unified_samples(record: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Build conversational message formats for a single-pass joint-output model.
    It splits (flattens) multiple questions into separate training samples.
    """

    premises = record.get("premises", [])
    premises_fol = record.get("premises-FOL", [])
    
    questions = record.get("question", [])
    options_list = record.get("options", [])
    answers = record.get("answer", [])
    explanations = record.get("explanation", [])
    idx_list = record.get("idx", [])
    z3_codes = record.get("z3_code", [])
    # 1. System Prompt mới thiết kế cho câu hỏi đơn lẻ (Singular XML)
    system_prompt = (
        "You are a logical reasoning expert. Given a set of premises in natural language, a query, and a list of options:\n"
        "1. You must first provide a concise step-by-step reasoning explanation inside the <think>...</think> tags.\n"
        "2. After closing the </think> tag, output the finalized structured results using XML tags:\n"
        "   - Use <premises_fol>...</premises_fol> to list the FOL translations of all premises.\n"
        "   - Use <question_output>...</question_output> containing the structured output for the query.\n"
        "   - Within the question block, provide the pedagogical <explanation> for the user, the list of <premises_used>, the executable <z3_code>, and the final <answer>.\n\n"
        "Important Guidelines:\n"
        "- In the <explanation> tag, always explain step-by-step how the answer is derived and explicitly cite the premise numbers you used (e.g., 'From premise 3, we know that...'). Do not refer to premises without their index numbers.\n"
        "- Ensure that you use consistent variable names in the Z3 code corresponding to the entities and relations in the premises.\n"
        "- If Options is non-empty, your final answer in the <answer> tag must match exactly one of the listed options. If Options is empty ([]), the answer is free-form (a number or a short text)."
    )
    formatted_premises = format_nl_premises(premises)
    formatted_premises_fol = format_fol_premises(premises_fol)
    
    samples = []
    
    for i in range(len(questions)):
        query = questions[i]
        options = options_list[i]
        answer = answers[i]
        explanation = explanations[i]
        premises_used = idx_list[i]
        z3_code = z3_codes[i]
        
        user_content = (
            f"Premises:\n{formatted_premises}\n\n"
            f"Question: {query}\n"
            f"Options: {json.dumps(options, ensure_ascii=False)}"
        )
        assistant_content = (
            f"<think>\n{explanation.strip()}\n</think>\n\n"
            f"<premises_fol>\n{formatted_premises_fol}\n</premises_fol>\n\n"
            f"<question_output>\n"
            f"  <explanation>\n    {explanation.strip()}\n  </explanation>\n"
            f"  <premises_used>{json.dumps(premises_used)}</premises_used>\n"
            f"  <z3_code>\n{z3_code.strip()}\n  </z3_code>\n"
            f"  <answer>{answer.strip()}</answer>\n"
            f"</question_output>"
        )
        samples.append({
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": assistant_content}
            ]
        })
        
    return samples
def prepare_data(
    input_file: str,
    output_dir: str,
    train_ratio: float = 0.8,
    val_ratio: float = 0.1,
    test_ratio: float = 0.1,
    seed: int = 42
):
    """Load, process, shuffle and split dataset at the sample level (after flattening)."""
    print(f"Loading data from {input_file}...")
    with open(input_file, "r", encoding="utf-8") as f:
        records = json.load(f)
    # Biến đổi và tách nhỏ (flatten) tất cả các câu hỏi
    samples = []
    for rec in records:
        samples.extend(build_unified_samples(rec))
        
    print(f"Processed and flattened into {len(samples)} single-query samples successfully.")
    # Trộn ngẫu nhiên và chia tập (split)
    random.seed(seed)
    indices = list(range(len(samples)))
    random.shuffle(indices)
    total = len(samples)
    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)
    train_idx = indices[:train_end]
    val_idx = indices[train_end:val_end]
    test_idx = indices[val_end:]
    print(f"Split: Train={len(train_idx)}, Val={len(val_idx)}, Test={len(test_idx)}")
    # Tạo thư mục đầu ra
    os.makedirs(output_dir, exist_ok=True)
    splits = {
        "logic_train.jsonl": train_idx,
        "logic_val.jsonl": val_idx,
        "logic_test.jsonl": test_idx
    }
    for filename, idx_list in splits.items():
        filepath = os.path.join(output_dir, filename)
        with open(filepath, "w", encoding="utf-8") as f:
            for idx in idx_list:
                f.write(json.dumps(samples[idx], ensure_ascii=False) + "\n")
        print(f"Saved {len(idx_list)} samples to {filepath}")

In [3]:
workspace_dir = r"d:\Education\exact_2026"
input_json = os.path.join(workspace_dir, "data", "processed", "Logic_SFT_with_options.json")
output_directory = os.path.join(workspace_dir, "data", "processed")

prepare_data(
    input_file=input_json,
    output_dir=output_directory,
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    seed=42
)

Loading data from d:\Education\exact_2026\data\processed\Logic_SFT_with_options.json...
Processed and flattened into 1686 single-query samples successfully.
Split: Train=1348, Val=168, Test=170
Saved 1348 samples to d:\Education\exact_2026\data\processed\logic_train.jsonl
Saved 168 samples to d:\Education\exact_2026\data\processed\logic_val.jsonl
Saved 170 samples to d:\Education\exact_2026\data\processed\logic_test.jsonl
